# Logistic Regression — Bank Health Classification

Linear baseline model predicting `bank_condition` (Healthy / Stressed / Critical)
from the engineered bank-health features.

Key properties:

- **Scaling:** features are standardized (`StandardScaler`) — required for the
  linear model — fitted on the training data only, inside the pipeline.
- **Split:** scenario-grouped train/test split with a fixed seed, identical to the
  Random Forest and XGBoost notebooks, so all three models are comparable.
- **Class imbalance:** `class_weight="balanced"` compensates for the Healthy-heavy
  class distribution.
- **Diagnostics:** VIF (variance inflation factor) multicollinearity check.
- **Metrics:** accuracy, balanced accuracy, macro precision/recall/F1, Critical-class
  recall and ROC-AUC (macro one-vs-rest), all computed on the held-out test set.

## 1. Data Loading

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

PROCESSED_DIR = Path("../processed")
MODELS_DIR = Path("../models")
OUTPUT_DIR = Path("../output")

MODELS_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
CLASS_ORDER = ["Healthy", "Stressed", "Critical"]
TARGET_MAPPING = {"Healthy": 0, "Stressed": 1, "Critical": 2}

SHOCK_COLS = [
    "gdp_shock_pp",
    "unemp_shock_pp",
    "rate_shock_pp",
    "credit_spread_bps",
    "inflation_shock_pp",
    "fx_devaluation_pct",
]

In [2]:
features = pd.read_csv(PROCESSED_DIR / "bank_health_features.csv")

TARGET = "bank_condition"
DROP_COLS = ["bank_id", "scenario_id", TARGET, "bank_condition_code"]

X = features.drop(columns=DROP_COLS)
y = features[TARGET]
groups = features["scenario_id"]

print("Dataset shape:", features.shape)
print("Number of model features:", X.shape[1])
print("\nTarget distribution (%):")
print((y.value_counts(normalize=True) * 100).round(2))

Dataset shape: (20000, 18)
Number of model features: 14

Target distribution (%):
bank_condition
Healthy     51.46
Stressed    32.78
Critical    15.76
Name: proportion, dtype: float64


## 2. Preprocessing

### 2.1 Scenario-Grouped Train/Test Split

The dataset is a bank x scenario panel, so the split is grouped by `scenario_id`:
the same scenario never appears in both train and test. The split parameters and
seed below are identical in all three model notebooks, which guarantees every
model is trained and evaluated on exactly the same rows.

In [3]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_scenarios = set(groups.iloc[train_idx])
test_scenarios = set(groups.iloc[test_idx])

print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)
print("Training scenarios:", len(train_scenarios))
print("Testing scenarios :", len(test_scenarios))
print("Scenario overlap  :", train_scenarios & test_scenarios)

Train shape: (16000, 14) | Test shape: (4000, 14)
Training scenarios: 400
Testing scenarios : 100
Scenario overlap  : set()


### 2.2 Leakage-Free `shock_severity_score`

`shock_severity_score` in the feature file was min-max scaled over **all**
scenarios before any split — a mild form of data leakage (scaler statistics
computed on data that includes the test set). It is rebuilt here from the raw
shock columns so the min/max statistics come from the **training** scenarios
only and are then applied unchanged to the test scenarios.

In [4]:
panel_shocks = (
    pd.read_csv(
        PROCESSED_DIR / "bank_stress_simulated_panel_clean.csv",
        usecols=["scenario_id"] + SHOCK_COLS,
    )
    .drop_duplicates("scenario_id")
    .set_index("scenario_id")[SHOCK_COLS]
    .abs()
)

row_magnitudes = features[["scenario_id"]].join(panel_shocks, on="scenario_id")[SHOCK_COLS]
assert row_magnitudes.notna().all().all(), "Missing shock values for some scenarios"

train_mins = row_magnitudes.iloc[train_idx].min()
train_maxs = row_magnitudes.iloc[train_idx].max()

shock_severity = ((row_magnitudes - train_mins) / (train_maxs - train_mins)).mean(axis=1)

X_train["shock_severity_score"] = shock_severity.iloc[train_idx].values
X_test["shock_severity_score"] = shock_severity.iloc[test_idx].values

print("shock_severity_score rebuilt using train-only min/max scaling.")
print(X_train["shock_severity_score"].describe().round(4))

shock_severity_score rebuilt using train-only min/max scaling.
count    16000.0000
mean         0.4166
std          0.2356
min          0.0201
25%          0.2099
50%          0.4012
75%          0.6212
max          0.8932
Name: shock_severity_score, dtype: float64


### 2.3 Multicollinearity Check (VIF)

VIF diagnostic for the linear model, computed on the **training** data only
(median-imputed; scaling does not affect VIF). As a rule of thumb, VIF > 10
signals strong multicollinearity. This is a diagnostic check — all 14 features
are kept for the model fit below.

In [5]:
vif_imputer = SimpleImputer(strategy="median")
X_train_imputed = vif_imputer.fit_transform(X_train)


def variance_inflation_factor(matrix, col_idx):
    """VIF of one column via the R^2 of regressing it on all other columns."""
    target_col = matrix[:, col_idx]
    other_cols = np.delete(matrix, col_idx, axis=1)
    r_squared = LinearRegression().fit(other_cols, target_col).score(other_cols, target_col)
    return 1.0 / (1.0 - r_squared)


vif_table = pd.DataFrame(
    {
        "Feature": X_train.columns,
        "VIF": [
            variance_inflation_factor(X_train_imputed, i)
            for i in range(X_train.shape[1])
        ],
    }
).sort_values("VIF", ascending=False).reset_index(drop=True)

print(vif_table.round(2).to_string(index=False))
print("\nFeatures with VIF > 10:", vif_table.loc[vif_table["VIF"] > 10, "Feature"].tolist())

                 Feature   VIF
          severity_score 10.78
    shock_severity_score  9.59
      concentration_flag  6.44
concentration_x_severity  6.06
        bank_risk_factor  5.16
         risk_x_severity  5.07
       top_sector_weight  2.33
        baseline_roa_pct  1.50
        liquidity_buffer  1.45
       sector_risk_score  1.40
              car_buffer  1.39
  deposit_to_asset_ratio  1.31
              size_score  1.29
     loan_to_asset_ratio  1.28

Features with VIF > 10: ['severity_score']


## 3. Training

Pipeline: median imputation -> standardization -> balanced logistic regression.
All preprocessing steps are fitted on the training data only.

In [6]:
logistic_model = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

logistic_model.fit(X_train, y_train)
print("Logistic Regression training completed.")

Logistic Regression training completed.


## 4. Evaluation

All metrics are computed on the held-out test set. ROC-AUC is reported as a
macro average over the three one-vs-rest binary problems, which is appropriate
for the imbalanced multiclass setting.

In [7]:
y_pred = logistic_model.predict(X_test)

classes = list(logistic_model.named_steps["classifier"].classes_)
proba = logistic_model.predict_proba(X_test)[:, [classes.index(c) for c in CLASS_ORDER]]

report = classification_report(y_test, y_pred, labels=CLASS_ORDER, output_dict=True)

test_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
    "Precision (macro)": precision_score(
        y_test, y_pred, average="macro", labels=CLASS_ORDER, zero_division=0
    ),
    "Recall (macro)": recall_score(
        y_test, y_pred, average="macro", labels=CLASS_ORDER, zero_division=0
    ),
    "F1 (macro)": f1_score(
        y_test, y_pred, average="macro", labels=CLASS_ORDER, zero_division=0
    ),
    "Recall (Critical)": report["Critical"]["recall"],
    "ROC-AUC (macro OVR)": roc_auc_score(
        y_test.map(TARGET_MAPPING),
        proba,
        multi_class="ovr",
        average="macro",
        labels=[0, 1, 2],
    ),
}

print(pd.Series(test_metrics).round(4).to_string())

Accuracy               0.8608
Balanced Accuracy      0.8610
Precision (macro)      0.8513
Recall (macro)         0.8610
F1 (macro)             0.8554
Recall (Critical)      0.8767
ROC-AUC (macro OVR)    0.9636


In [8]:
print(classification_report(y_test, y_pred, labels=CLASS_ORDER, digits=4))

confusion = confusion_matrix(y_test, y_pred, labels=CLASS_ORDER)
confusion_output = pd.DataFrame(
    confusion,
    index=[f"Actual_{c}" for c in CLASS_ORDER],
    columns=[f"Predicted_{c}" for c in CLASS_ORDER],
)
print(confusion_output)

              precision    recall  f1-score   support

     Healthy     0.9341    0.8764    0.9043      1974
    Stressed     0.7745    0.8300    0.8013      1353
    Critical     0.8453    0.8767    0.8607       673

    accuracy                         0.8608      4000
   macro avg     0.8513    0.8610    0.8554      4000
weighted avg     0.8652    0.8608    0.8621      4000

                 Predicted_Healthy  Predicted_Stressed  Predicted_Critical
Actual_Healthy                1730                 244                   0
Actual_Stressed                122                1123                 108
Actual_Critical                  0                  83                 590


## 5. Save Artifacts

Saves the fitted pipeline (imputer + scaler + classifier) plus the test-set
outputs: predictions vs actuals with class probabilities, and the metrics table
consumed by the model comparison notebook.

In [9]:
model_path = MODELS_DIR / "linear_regression_model.pkl"
joblib.dump(logistic_model, model_path)

predictions_output = pd.DataFrame(
    {
        "bank_id": features.iloc[test_idx]["bank_id"].values,
        "scenario_id": features.iloc[test_idx]["scenario_id"].values,
        "actual_condition": y_test.values,
        "predicted_condition": y_pred,
        "prob_healthy": proba[:, 0],
        "prob_stressed": proba[:, 1],
        "prob_critical": proba[:, 2],
    }
)
predictions_path = OUTPUT_DIR / "logistic_regression_predictions.csv"
predictions_output.to_csv(predictions_path, index=False)

metrics_output = pd.DataFrame(
    {"Metric": list(test_metrics), "Score": list(test_metrics.values())}
)
metrics_path = OUTPUT_DIR / "logistic_regression_metrics.csv"
metrics_output.to_csv(metrics_path, index=False)

print("Saved:", model_path)
print("Saved:", predictions_path)
print("Saved:", metrics_path)

Saved: ..\models\linear_regression_model.pkl
Saved: ..\output\logistic_regression_predictions.csv
Saved: ..\output\logistic_regression_metrics.csv
